In [7]:
import time
from kor.extraction import create_extraction_chain
from kor.nodes import Object, Text
from langchain.prompts import PromptTemplate
from kor import JSONEncoder
from langchain.llms import Ollama


# -------------------- CONFIGURATION --------------------

class EncoderAnalyze(JSONEncoder):
    def get_instruction_segment(self) -> str:
        return "Génère l'output sous le format JSON.\n"


# -------------------- CLASSIFICATEUR --------------------

class FewShotClassification():
    def __init__(self, llm):
        self.llm = llm
        self.promt_description = """[INST] Vous êtes un classificateur d'intention. Vous suivez extrêmement bien les instructions. \n
Votre objectif est de classifier des phrases correspondant à des requêtes/demandes d'un utilisateur qui vous seront fournies selon les 3 catégories suivantes : 
'Intention de recherche d'informations', 'Intention familière' et 'Intention d'actions'. \n
La requête de l'utilisateur peut-être formulée avec un ton formel, informel ou naturel. Des exemples de requêtes/demandes avec leur classification sont fournis pour vous donner une idée.
"""

    def generate(self, clause):
        s = time.perf_counter()
        chain = self._get_chain()
        result = chain.invoke(input=self._format_clause(clause))
        elapsed = time.perf_counter() - s
        return result, elapsed, chain

    def _format_clause(self, clause):
        return "Voici la phrase à classifier :\n" + clause

    def _get_chain(self):
        instruction_template = self.get_prompt()
        output_schema = self._get_expected_schema()
        return create_extraction_chain(
            self.llm,
            output_schema,
            instruction_template=instruction_template,
            encoder_or_encoder_class=EncoderAnalyze
        )

    def get_prompt(self):
        context = "Ne pas ajouter d'attributs ou de texte supplémentaires. Classer uniquement la requête de l'utilisateur dans l'une des 3 catégories fournies.\n[/INST]."
        return PromptTemplate(
            input_variables=["format_instructions", "type_description"],
            template=(
                f"{self.promt_description}\n\n"
                "{type_description}\n\n"
                "{format_instructions}\n"
                f"{context}\n\n"
            )
        )

    def _get_expected_schema(self):
        schema = Object(
            id="items",
            description="Information concernant la classification de phrases",
            attributes=[
                Text(
                    id="categorie",
                    description="Correspond au label (la catégorie) à laquelle appartient la requête"
                ),
                Text(
                    id="commentaire",
                    description="Explication/justification très succinte de classe choisie"
                )
            ],
            examples=[
                (
                    "Pourriez-vous me fournir les dernières directives concernant les procédures de sécurité ?",
                    [{"categorie": "Intention de recherche d'information", "commentaire": "Il cherche des directives, donc des informations"}],
                ),
                (
                    "Tu pourrais me dire si je me tiens mal pour éviter le mal de dos ?",
                    [{"categorie": "Intention familière", "commentaire": "Aucune recherche d'informations. Il s'agit d'une intention purement familière"}],
                ),
                (
                    "Engagez le protocole de sécurité pour verrouillage automatique des portes.",
                    [{"categorie": "Intention d'actions", "commentaire": "Intention d'action car requête relevant d'une action à faire"}],
                ),
                (
                    "J'ai besoin de choper les critères de qualité qu'on doit suivre, tu as ça ?",
                    [{"categorie": "Intention de recherche d'information", "commentaire": "Il cherche les critères de qualité"}],
                ),
                (
                    "Checke si on respire de l'air clean ici.",
                    [{"categorie": "Intention d'actions", "commentaire": "Intention d'action car requête relevant d'une action à faire"}],
                ),
                (
                    "Je requiers une analyse de mon niveau de stress au cours de la journée de travail.",
                    [{"categorie": "Intention familière", "commentaire": "Intention familière car question sur sa personne"}],
                ),
                (
                    "Ferme automatiquement les portes à l'heure prévue.",
                    [{"categorie": "Intention d'actions", "commentaire": "Intention d'action car requête relevant d'une action à faire"}],
                ),
                (
                    "Tu as les performances des équipements des derniers mois ?",
                    [{"categorie": "Intention de recherche d'information", "commentaire": "Cherche les performances"}],
                ),
                (
                    "Il me faudrait un retour sur mon stress pendant le boulot.",
                    [{"categorie": "Intention familière", "commentaire": "Sujet familier concernant l'utilisateur donc intention familière"}],
                ),
                (
                    "Quoi de neuf dans le monde des robots ?",
                    [{"categorie": "Intention de recherche d'information", "commentaire": "Ton familier et naturel mais recherche d'informations"}],
                ),
                (
                    "Veuillez activer le système d'alerte en cas de détection de fumée.",
                    [{"categorie": "Intention d'actions", "commentaire": "Demande explicite de réalisation d'une action"}],
                ),
                (
                    "Pouvez-vous me rappeler de faire des pauses régulières pour éviter la surcharge cognitive ?",
                    [{"categorie": "Intention familière", "commentaire": "Question familière pour faire des pauses"}],
                ),
                (
                    "Des idées pour me rappeler de boire suffisamment ?",
                    [{"categorie": "Intention familière", "commentaire": "Il veut être rappelé d'un sujet le concernant directement donc intention familière"}],
                ),
                (
                    "Fais tourner le check-up des machines pour voir si tout roule.",
                    [{"categorie": "Intention d'actions", "commentaire": "Demande explicite de vérification donc intention d'action"}],
                ),
                (
                    "Comment on fait pour la maintenance des robots, t'as un tuto ou un truc du genre ?",
                    [{"categorie": "Intention de recherche d'information", "commentaire": "Il cherche à savoir comment on fait la maintenance"}],
                ),
                (
                    "Quelles sont les recommandations pour l'optimisation des flux de travail ?",
                    [{"categorie": "Intention de recherche d'information", "commentaire": "Il veut avoir des infos sur les recommandations"}],
                ),
            ]
        )
        return schema



In [16]:
# Exemple de phrase à classifier
sentence = "Tu peux activer les capteurs thermiques dans les zones critiques ?"

# Utilisation de Mistral via Ollama
llm = Ollama(model="mistral")

classifier = FewShotClassification(llm)
output, elapsed, chain = classifier.generate(sentence)



print("\n--- PHRASE À CLASSIFIER ---")
print(sentence)

print("\n--- RÉSULTAT ---")
print(output)

print(f"\nTemps de génération : {elapsed:.2f} secondes")

# Récupérer le prompt template utilisé
prompt_template = classifier.get_prompt()
format_instructions = EncoderAnalyze().get_instruction_segment()
type_description = classifier._get_expected_schema().description

formatted_prompt = prompt_template.format(
    format_instructions=format_instructions,
    type_description=type_description
)
print("\n--- PROMPT FINAL ENVOYÉ À MISTRAL ---")
print(formatted_prompt)



--- PHRASE À CLASSIFIER ---
Tu peux activer les capteurs thermiques dans les zones critiques ?

--- RÉSULTAT ---
{'data': {'items': [{'categorie': "Intention d'actions", 'commentaire': "Demande explicite de réalisation d'une action"}]}, 'raw': ' <json>{"items": [{"categorie": "Intention d\'actions", "commentaire": "Demande explicite de réalisation d\'une action"}]}</json>', 'errors': [], 'validated_data': {}}

Temps de génération : 2.78 secondes

--- PROMPT FINAL ENVOYÉ À MISTRAL ---
[INST] Vous êtes un classificateur d'intention. Vous suivez extrêmement bien les instructions. 

Votre objectif est de classifier des phrases correspondant à des requêtes/demandes d'un utilisateur qui vous seront fournies selon les 3 catégories suivantes : 
'Intention de recherche d'informations', 'Intention familière' et 'Intention d'actions'. 

La requête de l'utilisateur peut-être formulée avec un ton formel, informel ou naturel. Des exemples de requêtes/demandes avec leur classification sont fournis p

In [18]:
import time
from kor.extraction import create_extraction_chain
from kor.nodes import Object, Text
from langchain.prompts import PromptTemplate
from kor import JSONEncoder
from langchain.llms import Ollama


# -------------------- CONFIGURATION --------------------

class EncoderAnalyze(JSONEncoder):
    def get_instruction_segment(self) -> str:
        return "Génère l'output sous le format JSON.\n"


# -------------------- CLASSIFICATEUR --------------------

class FewShotClassification():
    def __init__(self, llm):
        self.llm = llm
        self.promt_description = """[INST] Vous êtes un classificateur d'intention. Vous suivez extrêmement bien les instructions. \n
Votre objectif est de classifier des phrases correspondant à des requêtes/demandes d'un utilisateur qui vous seront fournies selon les 3 catégories suivantes : 
'Intention de recherche d'informations', 'Intention familière' et 'Intention d'actions'. \n
La requête de l'utilisateur peut-être formulée avec un ton formel, informel ou naturel. Des exemples de requêtes/demandes avec leur classification sont fournis pour vous donner une idée.
"""

    def generate(self, clause):
        s = time.perf_counter()
        chain = self._get_chain()
        result = chain.invoke(input=self._format_clause(clause))
        elapsed = time.perf_counter() - s
        return result, elapsed, chain

    def _format_clause(self, clause):
        return "Voici la phrase à classifier :\n" + clause

    def _get_chain(self):
        instruction_template = self.get_prompt()
        output_schema = self._get_expected_schema()
        return create_extraction_chain(
            self.llm,
            output_schema,
            instruction_template=instruction_template,
            encoder_or_encoder_class=EncoderAnalyze
        )

    def get_prompt(self):
        context = "Ne pas ajouter d'attributs ou de texte supplémentaires. Classer uniquement la requête de l'utilisateur dans l'une des 3 catégories fournies.\n[/INST]."
        return PromptTemplate(
            input_variables=["format_instructions", "type_description"],
            template=(
                f"{self.promt_description}\n\n"
                "{type_description}\n\n"
                "{format_instructions}\n"
                f"{context}\n\n"
            )
        )

    def _get_expected_schema(self):
        schema = Object(
            id="items",
            description="Détermine si une phrase est en rapport avec la situation actuelle des viticulteurs",
            attributes=[
                Text(
                    id="categorie",
                    description="Indique si la phrase est 'Pertinente' (en lien avec la situation actuelle des viticulteurs) ou 'Hors sujet'"
                ),
                Text(
                    id="commentaire",
                    description="Brève explication justifiant la classification"
                )
            ],
            examples=[
                (
                    "Les viticulteurs subissent de plein fouet les conséquences du gel printanier cette année.",
                    [{"categorie": "Pertinente", "commentaire": "Fait directement référence à des difficultés actuelles des viticulteurs"}],
                ),
                (
                    "La météo s’annonce ensoleillée toute la semaine dans le sud de la France.",
                    [{"categorie": "Hors sujet", "commentaire": "Ne traite pas des viticulteurs ou de leur situation"}],
                ),
                (
                    "Une aide exceptionnelle a été débloquée pour les exploitants touchés par la sécheresse.",
                    [{"categorie": "Pertinente", "commentaire": "Traite d’un soutien aux viticulteurs en difficulté"}],
                ),
                (
                    "Le vin rouge se marie très bien avec le fromage de brebis.",
                    [{"categorie": "Hors sujet", "commentaire": "Relatif au vin, mais pas à la situation actuelle des viticulteurs"}],
                ),
                (
                    "Manifestation des vignerons en Bourgogne pour dénoncer la baisse des prix.",
                    [{"categorie": "Pertinente", "commentaire": "Fait allusion à une revendication actuelle des viticulteurs"}],
                ),
                (
                    "Les vendanges ont commencé avec deux semaines d’avance cette année.",
                    [{"categorie": "Pertinente", "commentaire": "Indique un changement inhabituel dans le calendrier des viticulteurs"}],
                ),
                (
                    "Saviez-vous que le pinot noir est originaire de Bourgogne ?",
                    [{"categorie": "Hors sujet", "commentaire": "Fait historique sans rapport avec une situation actuelle"}],
                ),
            ]
        )
        return schema




In [21]:
# Exemple de phrase à classifier
sentence = "La technologie évolue plus vite que jamais ! 🚀 Plongez dans le monde de l'#AI et découvrez comment l'#Innovation façonne notre avenir. Prêt à rejoindre la révolution ? Construisons demain, dès aujourd'hui ! 💡✨"

# Utilisation de Mistral via Ollama
llm = Ollama(model="mistral")

classifier = FewShotClassification(llm)
output, elapsed, chain = classifier.generate(sentence)



print("\n--- PHRASE À CLASSIFIER ---")
print(sentence)

print("\n--- RÉSULTAT ---")
print(output)

print(f"\nTemps de génération : {elapsed:.2f} secondes")

# Récupérer le prompt template utilisé
prompt_template = classifier.get_prompt()
format_instructions = EncoderAnalyze().get_instruction_segment()
type_description = classifier._get_expected_schema().description

formatted_prompt = prompt_template.format(
    format_instructions=format_instructions,
    type_description=type_description
)
print("\n--- PROMPT FINAL ENVOYÉ À MISTRAL ---")
print(formatted_prompt)



--- PHRASE À CLASSIFIER ---
La technologie évolue plus vite que jamais ! 🚀 Plongez dans le monde de l'#AI et découvrez comment l'#Innovation façonne notre avenir. Prêt à rejoindre la révolution ? Construisons demain, dès aujourd'hui ! 💡✨

--- RÉSULTAT ---
{'data': {'items': [{'categorie': 'Hors sujet', 'commentaire': 'Ne traite pas des viticulteurs ou de leur situation'}]}, 'raw': ' <json>{"items": [{"categorie": "Hors sujet", "commentaire": "Ne traite pas des viticulteurs ou de leur situation"}]}</json>', 'errors': [], 'validated_data': {}}

Temps de génération : 2.26 secondes

--- PROMPT FINAL ENVOYÉ À MISTRAL ---
[INST] Vous êtes un classificateur d'intention. Vous suivez extrêmement bien les instructions. 

Votre objectif est de classifier des phrases correspondant à des requêtes/demandes d'un utilisateur qui vous seront fournies selon les 3 catégories suivantes : 
'Intention de recherche d'informations', 'Intention familière' et 'Intention d'actions'. 

La requête de l'utilisateu